In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.metrics import r2_score

# 1. Reconstruct the complete dataset
data = {
    'model': [
        'llama-3.1-8b-instruct-turbo',
        'llama-3.2-11b-vision-instruct-turbo',
        'llama-2-7b',
        'llama-2-13b',
        'llama-3-8b',
        'llama-2-70b',
        'llama-3-70b',
        'llama-3.1-70b-instruct-turbo',
        'llama-3.2-90b-vision-instruct-turbo',
        'llama-3.3-70b-instruct-turbo'
    ],
    'f2_raw': [-1.343, -1.359, 0.645, 0.645, 0.662, 1.343, 2.015, 2.265, 2.752, 2.774],
    'squad_f1': [89.51, 90.64, 69.57, 78.09, 90.44, 73.16, 91.96, 91.54, 91.70, 91.23],
    'mmlu': [6953.0, 6989.0, 5738.0, 7126.0, 8502.0, 9018.0, 10346.0, 10667.0, 10692.0, 10466.0],
    'math': [322.0, 336.0, 45.0, 56.0, 186.0, 159.0, 312.0, 364.0, 367.0, 370.0],
    'gsm': [798.0, 823.0, 154.0, 266.0, 499.0, 567.0, 805.0, 938.0, 936.0, 942.0],
    'commonsense': [369.0, 360.0, 272.0, 316.0, 382.0, 418.0, 465.0, 467.0, 469.0, 462.0],
    'size_b': [8, 11, 7, 13, 8, 70, 70, 70, 90, 70]
}
df = pd.DataFrame(data)

# 2. Define the list of proxies and initialize a list to store results
proxies = ['mmlu', 'math', 'gsm', 'commonsense', 'size_b']
results_list = []

# 3. Loop through each proxy to perform the analysis
for proxy_name in proxies:
    # --- Prepare data ---
    X_df = df[[proxy_name]]
    y = df['squad_f1']

    # Use log transformation for model size, as capability scales non-linearly
    if proxy_name == 'size_b':
        X_lin = np.log(X_df)
        X_loess = np.log(X_df.values.flatten())
    else:
        X_lin = X_df
        X_loess = X_df.values.flatten()

    # --- Linear Fit Analysis ---
    lin_reg = LinearRegression().fit(X_lin, y)
    linear_pred = lin_reg.predict(X_lin)
    r2_lin = r2_score(y, linear_pred)
    aptitude_score_lin = y - linear_pred
    corr_lin = df['f2_raw'].corr(aptitude_score_lin)

    # --- LOESS (Non-linear) Fit Analysis ---
    # frac=0.6 provides a good balance of smoothness and fit for this small dataset
    loess_pred = lowess(y, X_loess, frac=0.6, return_sorted=False)
    r2_loess = r2_score(y, loess_pred)
    aptitude_score_loess = y - loess_pred
    corr_loess = df['f2_raw'].corr(aptitude_score_loess)

    # --- Store results for the current proxy ---
    results_list.append({
        'Proxy Benchmark': proxy_name.upper(),
        'Linear Fit R²': r2_lin,
        'LOESS Fit R²': r2_loess,
        'Nonlinear Improvement': r2_loess - r2_lin,
        'F2 Corr w/ Linear Residual': corr_lin,
        'F2 Corr w/ LOESS Residual': corr_loess
    })

# 4. Create and display the final summary DataFrame
summary_df = pd.DataFrame(results_list)

# Set display options for better readability
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print("--- Summary of Aptitude Analysis ---")
print(summary_df.to_string())

--- Summary of Aptitude Analysis ---
  Proxy Benchmark  Linear Fit R²  LOESS Fit R²  Nonlinear Improvement  F2 Corr w/ Linear Residual  F2 Corr w/ LOESS Residual
0            MMLU         0.3305        0.5716                 0.2411                     -0.4129                    -0.3558
1            MATH         0.7548        0.6664                -0.0884                     -0.1121                    -0.0294
2             GSM         0.6897        0.8618                 0.1721                     -0.1815                    -0.0599
3     COMMONSENSE         0.4443        0.0311                -0.4132                     -0.4195                    -0.7690
4          SIZE_B         0.0800        0.0950                 0.0150                     -0.1010                    -0.1123


# Token Analysis

In [5]:
import spacy
import pandas as pd

# ensure the small English model is available; download if missing
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def compute_text_complexity(text):
    doc = nlp(text)
    n_tokens = len(doc)
    return {
        "n_tokens": n_tokens,
        "n_sentences": len(list(doc.sents)),
        "n_entities": len(doc.ents),
        "entity_types": len(set([ent.label_ for ent in doc.ents])),
        "avg_dep_depth": sum(len([t for t in tok.subtree]) for tok in doc) / n_tokens if n_tokens else 0.0,
        "n_pronouns": sum(1 for tok in doc if tok.pos_ == "PRON"),
    }

df = pd.read_csv('../output/conv_item_loadings_rotated_geomin_obl.csv')

df_eval = pd.DataFrame({
    "item": df['question'],
    "f2_loading": df['Factor_2']
})

df_eval_features = df["question"].apply(compute_text_complexity).apply(pd.Series)
df_eval = pd.concat([df_eval, df_eval_features], axis=1)

In [6]:
df_eval.select_dtypes(include=[float, int]).corr()["f2_loading"].sort_values()

avg_dep_depth   0.2811
n_entities      0.4070
entity_types    0.4357
n_pronouns      0.4652
n_sentences     0.4662
n_tokens        0.4962
f2_loading      1.0000
Name: f2_loading, dtype: float64